In [2]:
# Vamonet refined clustering, trying to do two subsequent clustering via vmaonet
# attempt to seperate data prior to inputting in vampnet

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis import contacts
import io
from collections import defaultdict
import itertools
import pickle
import seaborn as sns
import mdtraj as md
from itertools import islice
from deeptime.util.data import TrajectoryDataset, TrajectoriesDataset
from deeptime.util.torch import MLP
from deeptime.decomposition.deep import VAMPNet 
from deeptime.decomposition import VAMP
from deeptime.util.validation import implied_timescales, ck_test
from deeptime.plots import plot_implied_timescales, plot_ck_test
from torch.utils.data import DataLoader
import deeptime as dt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.cluster import adjusted_mutual_info_score
import itertools

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")
torch.set_num_threads(12)

from deeptime.data import sqrt_model


/Users/adelielouet/miniforge3/envs/msm/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Reproducibility of VAMPNet clustering across five independent training runs

Plot the Adjusted Mutual Information (AMI) scores between clustering outcomes, quantifying the consistency of state assignments.

In [3]:

xtc=('/Users/adelielouet/Documents/science/AB_G5_original_simu_analysis/trajectories/Gabis_paper/traj_all-skip-0-noW_G5.xtc')
pdb=('/Users/adelielouet/Documents/science/AB_G5_original_simu_analysis/trajectories/Gabis_paper/template_G5.pdb')



t = md.load(xtc, top=pdb)
topology = t.topology

ligand_atom_indices = [636, 628, 650]
protein_ca_indices = t.top.select("protein and name CA")
table, bonds = topology.to_dataframe()

file = open('/Users/adelielouet/Documents/science/AB_G5_original_simu_analysis/trajectories/Gabis_paper/distances_residue_com_liga.pickle','rb')
distances = pickle.load(file)
file.close()

distance_t_40=(np.stack(distances, axis=1))
distance_t_40=distance_t_40.reshape(256128, 42)


## Clustering number ONE: dividing trajecotry into metastates

#input vairables
lag=10
nstates=4
lr=5e-3
epochs=30  #n_epochs=nb_epoch: Specifies the number of training epochs (iterations over the entire dataset).
batch_size=10000

asssingnments_dict={}
for run_number in range(0,5):
    normalized_features = distance_t_40#distance_features_t_120#distance_features_t_120 / distance_features_t_120.sum(axis=1, keepdims=True)
    tensor_data = torch.tensor(normalized_features) #distance_features_t_120)
    # Step 3: Detach from the graph and move to CPU if necessary
    Xi = tensor_data.detach().cpu()
    # Step 4: Convert to NumPy array and wrap it in a list
    data1 = [Xi.numpy()]
    dataset = TrajectoriesDataset.from_numpy(lag, data1)   # define some lag time
    #dataset = dt.util.data.TrajectoryDataset(lagtime=tau, trajectory=data.astype(np.float32)) 
    n_val = int(len(dataset)*.1) # Portion of data set as test set
    train_data, val_data = torch.utils.data.random_split(dataset, [len(dataset) - n_val, n_val])    
    d=nstates-1 # define nstates before; d=number of hidden layers in the network
    dimsize = torch.ones(d+2,dtype=int) #size of each layer of the network
    n_in = data1[0].shape[1] #n_in = number of inputs for the network
    n_out = nstates # n_out= number of nodes in output layer = number of states
    frac = 1/np.power((n_in/n_out),1/d) # low network model should be chosen
    dimsize[0]= data1[0].shape[1] # n_in
    dimsize[1] = int(np.ceil(frac*n_in)) # 1/d
    dimsize[d+1] = nstates 

    for h in range(2,d+1):
        dimsize[h] = int(np.ceil(frac*dimsize[h-1]))  # set nodes for each layer
    print(n_in,n_out,dimsize)
    lobe1 = MLP(units=dimsize, nonlinearity=nn.ELU,initial_batchnorm=True) # Define a lobe network
    lobe = nn.Sequential(lobe1,nn.Softmax(dim=1)) # Output layer of the lobe ; fuzzy clustering
    lobe = lobe.to(device=device) 

    vampnet = VAMPNet(lobe=lobe, learning_rate=lr, device=device)  # lr = learning rate has to be defined; to start with 5e-3 is good

    loader_train = DataLoader(train_data, batch_size=batch_size, shuffle=True)   #Batch size should be changed; better results with batchsize between 4000-10000
    loader_val = DataLoader(val_data, batch_size=len(val_data), shuffle=False)

    model = vampnet.fit(loader_train, n_epochs=epochs,
                    validation_loader=loader_val, progress=tqdm).fetch_model()


    state_probabilities = model.transform(data1[0])   #converts input data into fuzzy clusters
    assignments = state_probabilities.argmax(1) #gets discrete clusters from fuzzy probabilities
    lagtimes = np.arange(5, 201 ,5, dtype=np.int32) # this range can be changed; try different possibilities
    vamp_models = [VAMP(lagtime=lag, observable_transform=model).fit_fetch(data1) for lag in tqdm(lagtimes)] 
    its_data = implied_timescales(vamp_models)  #gets timescales out; might be meaningless for clustering IDP binding; but worth checking how it varies over multiple trials

    state_numbers=dict(pd.value_counts(assignments))
    print(f'The state values are: {state_numbers}')

    ### This is to add the ts to each assignemnt
    frames_assignments_labeled = {key: [] for key in set(assignments)}

    for timestep, (cluster, feature) in enumerate(zip(assignments, normalized_features)):
        frames_assignments_labeled[cluster].append((feature, timestep))

    ## Clustering number two: dividing trajecotry into microstates:

    frames_assignments = {key: [] for key in set(assignments)}
    for x, y in zip(assignments, normalized_features):
        frames_assignments[x].append(y)

    lag=5
    nstates=4
    lr=5e-3
    epochs=30  #n_epochs=nb_epoch: Specifies the number of training epochs (iterations over the entire dataset).
    batch_size=500
    count=0
    assignments_16_states=[]
    values_16_states=[]
    ts_seperated=[]

    for x in (range(0,nstates)):
        tensor_data = torch.tensor(frames_assignments[x]) 

        Xi = tensor_data.detach().cpu()
        data1 = [Xi.numpy()]
        dataset = TrajectoriesDataset.from_numpy(lag, data1) 

        n_val = int(len(dataset)*.1) # Portion of data set as test set
        train_data, val_data = torch.utils.data.random_split(dataset, [len(dataset) - n_val, n_val])    
        d=nstates-1 # define nstates before; d=number of hidden layers in the network
        dimsize = torch.ones(d+2,dtype=int) #size of each layer of the network
        n_in = data1[0].shape[1] #n_in = number of inputs for the network
        n_out = nstates # n_out= number of nodes in output layer = number of states
        frac = 1/np.power((n_in/n_out),1/d) # low network model should be chosen
        dimsize[0]= data1[0].shape[1] # n_in
        dimsize[1] = int(np.ceil(frac*n_in)) # 1/d
        dimsize[d+1] = nstates 

        for h in range(2,d+1):
            dimsize[h] = int(np.ceil(frac*dimsize[h-1]))  # set nodes for each layer
        lobe1 = MLP(units=dimsize, nonlinearity=nn.ELU,initial_batchnorm=True) # Define a lobe network
        lobe = nn.Sequential(lobe1,nn.Softmax(dim=1)) # Output layer of the lobe ; fuzzy clustering
        lobe = lobe.to(device=device) 
        #print(lobe)
        # lobe.to(device=device)
        #print(lobe)
        vampnet = VAMPNet(lobe=lobe, learning_rate=lr, device=device)  # lr = learning rate has to be defined; to start with 5e-3 is good

        loader_train = DataLoader(train_data, batch_size=batch_size, shuffle=True)   #Batch size should be changed; better results with batchsize between 4000-10000
        loader_val = DataLoader(val_data, batch_size=len(val_data), shuffle=False)

        model = vampnet.fit(loader_train, n_epochs=epochs,
                        validation_loader=loader_val, progress=tqdm).fetch_model()

        state_probabilities = model.transform(data1[0])   #converts input data into fuzzy clusters
        assignments = state_probabilities.argmax(1) #gets discrete clusters from fuzzy probabilities
        lagtimes = np.arange(5, 201 ,5, dtype=np.int32) # this range can be changed; try different possibilities
        vamp_models = [VAMP(lagtime=lag, observable_transform=model).fit_fetch(data1) for lag in tqdm(lagtimes)] 
        its_data = implied_timescales(vamp_models)  #gets timescales out; might be meaningless for clustering IDP binding; but worth checking how it varies over multiple trials

        assignments_plus_4 = [x + count for x in assignments]
        print(set(assignments_plus_4),set(assignments))
        print(f'count is {count}')
        count+=4

        assignments_16_states.append(assignments_plus_4)
        values_16_states.append(frames_assignments[x])

    assignments_16_states = list(itertools.chain(*assignments_16_states))

    values_16_states_flat = np.array(list(itertools.chain.from_iterable(values_16_states)))
    values_16_states_flat_reversed = values_16_states_flat.reshape(len(distance_t_40), len(distance_t_40[0])) #for com

    ts_shortcut=[]
    for keys,values in frames_assignments_labeled.items():
        for o in values:
            ts_shortcut.append(o[1])

    frames_assignments_labeled_16 = {key: [] for key in set(assignments_16_states)}

    
    frames_assignments_labeled_16 = {key: [] for key in set(assignments_16_states)}

    for (cluster, ts) in zip(assignments_16_states, ts_shortcut):
        frames_assignments_labeled_16[cluster].append(ts)

    clusters_dict=frames_assignments_labeled_16.copy()
    print(f'round {run_number} done')

    max_frame = max(frame for frames in clusters_dict.values() for frame in frames)

    assignments_16_states = [-1] * (max_frame + 1)
    for cluster, frames in clusters_dict.items():
        for frame in frames:
            assignments_16_states[frame] = int(cluster) 

    asssingnments_dict[run_number]=assignments_16_states


ami_scores = []
for i, j in itertools.combinations(range(len(asssingnments_dict)), 2):
    score = adjusted_mutual_info_score(asssingnments_dict[i], asssingnments_dict[j])
    ami_scores.append(score)

sns.kdeplot(ami_scores)
plt.xlabel('Score')
#plt.savefig(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/figures/vampnet_ami.png', dpi=300)
plt.show()


/var/folders/y3/sm2gn5053mz_rtfqrj0r8mz40000gn/T/ipykernel_54292/4068278214.py:14: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  distances = pickle.load(file)
/var/folders/y3/sm2gn5053mz_rtfqrj0r8mz40000gn/T/ipykernel_54292/4068278214.py:52: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  dimsize[h] = int(np.ceil(frac*dimsize[h-1]))  # set nodes for each layer


42 4 tensor([42, 20, 10,  5,  4])


100%|██████████| 40/40 [00:01<00:00, 25.56it/s]               
/var/folders/y3/sm2gn5053mz_rtfqrj0r8mz40000gn/T/ipykernel_54292/4068278214.py:73: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  state_numbers=dict(pd.value_counts(assignments))
/var/folders/y3/sm2gn5053mz_rtfqrj0r8mz40000gn/T/ipykernel_54292/4068278214.py:99: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1750198230540/work/torch/csrc/utils/tensor_new.cpp:257.)
  tensor_data = torch.tensor(frames_assignments[x])
/var/folders/y3/sm2gn5053mz_rtfqrj0r8mz40000gn/T/ipykernel_54292/4068278214.py:117: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.

The state values are: {2: np.int64(243152), 1: np.int64(5611), 3: np.int64(3997), 0: np.int64(3368)}


100%|██████████| 40/40 [00:00<00:00, 394.77it/s]              


...

100%|██████████| 40/40 [00:00<00:00, 284.47it/s]              


{np.int64(5), np.int64(6), np.int64(7)} {np.int64(1), np.int64(2), np.int64(3)}
count is 4


100%|██████████| 40/40 [00:01<00:00, 25.53it/s]               


{np.int64(8), np.int64(9), np.int64(10), np.int64(11)} {np.int64(0), np.int64(1), np.int64(2), np.int64(3)}
count is 8


KeyError: 3